In [2]:
import nltk
import zipfile
import os
from nltk.corpus import reuters
from nltk import bigrams, trigrams
from collections import Counter, defaultdict

# 1. Download ALL necessary resources
nltk.download('reuters')
nltk.download('punkt')
nltk.download('punkt_tab') # <--- This fixes the "No sentence tokenizer" error

# 2. Fix for Reuters in Colab
# (Reuters often downloads as a ZIP but doesn't unzip automatically in some envs)
try:
    # Try to access the dataset
    reuters.sents()
except (LookupError, OSError):
    print("Unzipping Reuters dataset...")
    # Find where it downloaded
    reuters_path = nltk.data.find('corpora/reuters.zip')
    # Unzip it
    with zipfile.ZipFile(reuters_path, 'r') as zip_ref:
        zip_ref.extractall(os.path.dirname(reuters_path))
    print("Unzip complete.")

print("Building N-Gram models... (This takes about 10 seconds)")

# 3. Train the Model
# We look at every sentence in the Reuters news dataset
corpus = reuters.sents()

# 'model' stores the counts: { (previous_word): {next_word: count} }
bigram_model = defaultdict(Counter)
trigram_model = defaultdict(Counter)

for sentence in corpus:
    # Build Bigrams (Context = 1 word)
    for w1, w2 in bigrams(sentence, pad_right=True, pad_left=True):
        bigram_model[w1][w2] += 1

    # Build Trigrams (Context = 2 words)
    for w1, w2, w3 in trigrams(sentence, pad_right=True, pad_left=True):
        trigram_model[(w1, w2)][w3] += 1

print("Training Complete! The model has read", len(corpus), "sentences.")

[nltk_data] Downloading package reuters to /root/nltk_data...
[nltk_data]   Package reuters is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Building N-Gram models... (This takes about 10 seconds)
Training Complete! The model has read 54716 sentences.


In [3]:
def predict_next_word(text, limit=3):
    """
    Predicts the next word based on the input text using N-grams.
    Priority: Trigrams (better context) -> Bigrams (back-off)
    """
    words = text.split()
    predictions = []

    # CASE 1: Use Trigram Model (if we have at least 2 words context)
    if len(words) >= 2:
        context = (words[-2], words[-1]) # Look at last 2 words
        if context in trigram_model:
            # Get most common next words
            top_words = trigram_model[context].most_common(limit)
            predictions = [(word, f"Trigram ({count})") for word, count in top_words]

    # CASE 2: Use Bigram Model (Back-off if Trigram fails or too few words)
    # If Trigram found nothing, or user only typed 1 word
    if not predictions and len(words) >= 1:
        context = words[-1] # Look at last 1 word
        if context in bigram_model:
            top_words = bigram_model[context].most_common(limit)
            predictions = [(word, f"Bigram ({count})") for word, count in top_words]

    return predictions

In [4]:
# Test Phrases
test_phrases = [
    "economic",          # 1 word context -> uses Bigram
    "the company",       # 2 word context -> uses Trigram
    "government has",
    "oil prices",
    "please do"
]

print(f"{'INPUT TEXT':<20} | {'SUGGESTIONS (Next Word)'}")
print("-" * 70)

for phrase in test_phrases:
    suggestions = predict_next_word(phrase)

    if suggestions:
        # Join the suggestions into a string
        suggestion_text = ", ".join([f"'{w[0]}'" for w in suggestions])
    else:
        suggestion_text = "(No suggestion found)"

    print(f"{phrase:<20} | {suggestion_text}")

INPUT TEXT           | SUGGESTIONS (Next Word)
----------------------------------------------------------------------
economic             | 'growth', 'fundamentals', 'policy'
the company          | ''', 'said', '.'
government has       | 'said', 'already', 'been'
oil prices           | '.', 'and', ','
please do            | 'not', 'so', 'it'
